# SECTION 1 – Initialize Spark Session & Environment Setup

This section initializes the Apache Spark session, configures application parameters, and loads essential utility libraries.

### Cell 1 – Install Prerequisites

In [1]:
!pip install pyspark gtfs-realtime-bindings protobuf pandas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Cell 2 – Import Core Libraries & Initialize Spark

In [2]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create Spark Session
spark = SparkSession.builder \
    .appName("Bus_Operator_Benchmarking") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print("Spark Version:", spark.version)

Spark Session Created Successfully!
Spark Version: 4.1.1


# SECTION 2 – Ingest Bus Services Dataset

This section loads raw bus service and timetable data into a PySpark DataFrame (`services_raw`).

### Cell 1 – Define File Path & Read Services Data

In [3]:
SERVICES_DATA_PATH = "data/bus_services.csv" # Update path as needed

# Read CSV with automated schema inference or string default
try:
    services_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(SERVICES_DATA_PATH)
except Exception as e:
    print("File load fallback / dummy schema definition:", e)

File load fallback / dummy schema definition: [PATH_NOT_FOUND] Path does not exist: file:/D:/big data assgn/code/data/bus_services.csv. SQLSTATE: 42K03


### Cell 2 – Verify Raw Dataset Schema and Sample

In [4]:
if 'services_raw' in locals():
    services_raw.printSchema()
    services_raw.show(5, truncate=False)

# SECTION 3 – Clean & Transform Services Dataset

This section performs data cleaning, Type casting, Null handling, and prepares `services_df` for operator benchmarking.

### Cell 1 – Clean Columns and Cast Data Types

In [5]:
# Ensure required numeric columns exist and null values are handled
if 'services_raw' in locals():
    services_df = services_raw \
        .withColumn("Trips", col("Trips").cast("integer")) \
        .withColumn("Stops", col("Stops").cast("integer")) \
        .na.fill({"Trips": 0, "Stops": 0})
    
    services_df.cache()
    print("Cleaned Services Count:", services_df.count())

# SECTION 4 – Create Operator Summary Table

This section aggregates comprehensive operator statistics (min/max/avg trips and stops), calculates a baseline performance score, runs SQL evaluations, and persists the dataset to Parquet and CSV files.

### Cell 1 – Load Functions

In [ ]:
from pyspark.sql.functions import *

### Cell 2 – Verify Dataset

In [ ]:
print("Rows :", services_df.count())
print("Columns :", len(services_df.columns))
services_df.show(5, truncate=False)

### Cell 3 – Calculate Operator Statistics

In [ ]:
operator_summary = (
    services_df
    .groupBy("Operator")
    .agg(
        count("*").alias("TotalServices"),
        countDistinct("ServiceCode").alias("UniqueServices"),
        countDistinct("LineName").alias("UniqueRoutes"),
        sum("Trips").alias("TotalTrips"),
        avg("Trips").alias("AverageTrips"),
        max("Trips").alias("MaximumTrips"),
        min("Trips").alias("MinimumTrips"),
        sum("Stops").alias("TotalStops"),
        avg("Stops").alias("AverageStops"),
        max("Stops").alias("MaximumStops"),
        min("Stops").alias("MinimumStops")
    )
)

### Cell 4 – View Results

In [ ]:
operator_summary.show(20, truncate=False)

### Cell 5 – Sort by Largest Operators

In [ ]:
operator_summary.orderBy(
    desc("TotalTrips")
).show(20, truncate=False)

### Cell 6 – Add Performance Score
This isn't the final ML score—it gives us a useful feature for later analysis.

In [ ]:
operator_summary = (
    operator_summary
    .withColumn(
        "PerformanceScore",
        round(
            col("TotalTrips") * 0.40 +
            col("TotalStops") * 0.30 +
            col("UniqueRoutes") * 15 +
            col("UniqueServices") * 10,
            2
        )
    )
)

### Cell 7 – View Scores

In [ ]:
operator_summary.select(
    "Operator",
    "PerformanceScore"
).orderBy(
    desc("PerformanceScore")
).show(20, truncate=False)

### Cell 8 – Register Spark SQL Table

In [ ]:
operator_summary.createOrReplaceTempView("operator_summary")

### Cell 9 – SQL Analysis

In [ ]:
spark.sql("""
SELECT
Operator,
TotalServices,
UniqueRoutes,
TotalTrips,
AverageTrips,
TotalStops,
AverageStops,
PerformanceScore
FROM operator_summary
ORDER BY PerformanceScore DESC
LIMIT 20
""").show(truncate=False)

### Cell 10 – Cache DataFrame

In [ ]:
operator_summary.cache()
operator_summary.count()

### Cell 11 – Check Partitions

In [ ]:
print("Partitions :", operator_summary.rdd.getNumPartitions())

### Cell 12 – Repartition

In [ ]:
operator_summary = operator_summary.repartition(8)
print("Partitions :", operator_summary.rdd.getNumPartitions())

### Cell 13 – Save as Parquet

In [ ]:
operator_summary.write.mode("overwrite").parquet("output/operator_summary")

### Cell 14 – Save as CSV

In [ ]:
operator_summary.toPandas().to_csv(
    "output/operator_summary.csv",
    index=False
)

### Cell 15 – Display Summary Statistics

In [ ]:
operator_summary.describe().show()

# SECTION 5 – Process Vehicle Location Dataset

This section ingests GTFS-Realtime (protobuf) vehicle positioning data, extracts vehicle/trip attributes, converts the payload into PySpark DataFrames, calculates route-level speed and location statistics, and persists the outputs.

### Cell 1 – Install GTFS Library (Run Once)

In [ ]:
!pip install gtfs-realtime-bindings protobuf

### Cell 2 – Import Libraries

In [ ]:
from google.transit import gtfs_realtime_pb2
import pandas as pd
from pyspark.sql.functions import *

### Cell 3 – Load GTFS-RT File
Update the path if your file is in a different location.

In [ ]:
GTFS_PATH = r"D:\big data assgn\gtfsrt_2026-07-28_094937\gtfsrt.bin"
feed = gtfs_realtime_pb2.FeedMessage()
with open(GTFS_PATH, "rb") as f:
    feed.ParseFromString(f.read())
print("Total GTFS Entities:", len(feed.entity))

### Cell 4 – Extract Vehicle Records

In [ ]:
vehicle_records = []
for entity in feed.entity:
    if entity.HasField("vehicle"):
        vehicle = entity.vehicle
        vehicle_records.append({
            "VehicleID": vehicle.vehicle.id,
            "TripID": vehicle.trip.trip_id,
            "RouteID": vehicle.trip.route_id,
            "Latitude": vehicle.position.latitude,
            "Longitude": vehicle.position.longitude,
            "Bearing": vehicle.position.bearing,
            "Speed": vehicle.position.speed,
            "Timestamp": vehicle.timestamp
        })
print("Vehicle Records:", len(vehicle_records))

### Cell 5 – Create Pandas DataFrame

In [ ]:
vehicle_df = pd.DataFrame(vehicle_records)
vehicle_df.head()

### Cell 6 – Convert to PySpark

In [ ]:
vehicle_spark = spark.createDataFrame(vehicle_df)
vehicle_spark.show(5, truncate=False)

### Cell 7 – Check Schema

In [ ]:
vehicle_spark.printSchema()

### Cell 8 – Remove Duplicate Records

In [ ]:
vehicle_spark = vehicle_spark.dropDuplicates()
print("Vehicle Records:", vehicle_spark.count())

### Cell 9 – Cache Dataset

In [ ]:
vehicle_spark.cache()
vehicle_spark.count()

### Cell 10 – Register SQL Table

In [ ]:
vehicle_spark.createOrReplaceTempView("vehicle_locations")

### Cell 11 – Route Statistics

In [ ]:
spark.sql("""
SELECT
RouteID,
COUNT(*) AS VehicleCount,
AVG(Speed) AS AverageSpeed,
MAX(Speed) AS MaximumSpeed
FROM vehicle_locations
GROUP BY RouteID
ORDER BY VehicleCount DESC
LIMIT 20
""").show(truncate=False)

### Cell 12 – Create Vehicle Summary

In [ ]:
vehicle_summary = (
    vehicle_spark
    .groupBy("RouteID")
    .agg(
        count("*").alias("VehicleCount"),
        avg("Speed").alias("AverageSpeed"),
        max("Speed").alias("MaximumSpeed"),
        min("Speed").alias("MinimumSpeed"),
        avg("Latitude").alias("AverageLatitude"),
        avg("Longitude").alias("AverageLongitude"),
        avg("Bearing").alias("AverageBearing")
    )
)

### Cell 13 – View Summary

In [ ]:
vehicle_summary.show(20, truncate=False)

### Cell 14 – Register SQL Table

In [ ]:
vehicle_summary.createOrReplaceTempView("vehicle_summary")

### Cell 15 – SQL Analysis

In [ ]:
spark.sql("""
SELECT *
FROM vehicle_summary
ORDER BY VehicleCount DESC
LIMIT 20
""").show(truncate=False)

### Cell 16 – Check Partitions

In [ ]:
print("Partitions :", vehicle_summary.rdd.getNumPartitions())

### Cell 17 – Repartition

In [ ]:
vehicle_summary = vehicle_summary.repartition(8)
print("Partitions :", vehicle_summary.rdd.getNumPartitions())

### Cell 18 – Persist Dataset

In [ ]:
from pyspark import StorageLevel
vehicle_summary.persist(StorageLevel.MEMORY_AND_DISK)
vehicle_summary.count()

### Cell 19 – Execution Plan

In [ ]:
vehicle_summary.explain(True)

### Cell 20 – Data Quality Check

In [ ]:
vehicle_summary.select(
    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in vehicle_summary.columns
    ]
).show()

### Cell 21 – Summary Statistics

In [ ]:
vehicle_summary.describe().show()

### Cell 22 – Save as Parquet

In [ ]:
vehicle_summary.write.mode("overwrite").parquet("output/vehicle_summary")

### Cell 23 – Save as CSV

In [ ]:
vehicle_summary.toPandas().to_csv(
    "output/vehicle_summary.csv",
    index=False
)